参考文献 :\
「SEMI-SUPERVISED CLASSIFICATION WITH GRAPH CONVOLUTIONAL NETWORKS」\
https://arxiv.org/pdf/1609.02907\
「PyTorch実装で理解するGraph Convolutional Network」\
https://zenn.dev/hash_yuki/articles/cb008a7c19e917 \
「Demystifying GCNs: A Step-by-Step Guide to Building a Graph Convolutional Network Layer in PyTorch」 \
https://medium.com/@jrosseruk/demystifying-gcns-a-step-by-step-guide-to-building-a-graph-convolutional-network-layer-in-pytorch-09bf2e788a51

目的 : \
Graph Convolutional Networks の基礎理解と実装を行う。\
実装はPytorchに接続する形で行い、入力/出力はPytorch Geometric に合わせることとする。

# インポート

In [1]:
import numpy as np
%matplotlib inline
from matplotlib import pyplot as plt
from typing import Any, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

# 基本的な実装

In [2]:
num_node = 5 # ノード数
in_channels = 4 # ノード特徴量の次元数
X = np.random.randn(num_node, in_channels) + 1 # ノード特徴量行列の生成（数値はランダム）

print(X.shape)
print(X)

(5, 4)
[[ 0.41124623  0.90770339  3.43693373  1.36181471]
 [ 1.91102225  1.23626986  0.53950581  0.85516361]
 [ 1.16425653  1.66615454 -0.06729404  0.44054956]
 [ 1.51757923  1.70288943  3.34446883  1.40439474]
 [ 0.24034553  0.5036889  -0.44573714  1.99709065]]


In [9]:
E = [[0, 1], [0, 2], [0, 4], [1, 2], [2, 3]] # エッジの定義
reversed_E = [[j, i] for [i, j] in E] # 無向エッジなので、逆向きのエッジを定義
I = [[i, i] for i in range(num_node)] # GCNは自己ループを追加する

new_E = E + reversed_E  + I # 足し合わせる
print(new_E)

[[0, 1], [0, 2], [0, 4], [1, 2], [2, 3], [1, 0], [2, 0], [4, 0], [2, 1], [3, 2], [0, 0], [1, 1], [2, 2], [3, 3], [4, 4]]


In [13]:
# エッジから隣接行列を作成する関数
def edge2mat(E, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in E:
        A[i, j] = 1
    return A

A_tilde = edge2mat(new_E, num_node)

print(A_tilde.shape)
print(A_tilde)

(5, 5)
[[1. 1. 1. 0. 1.]
 [1. 1. 1. 0. 0.]
 [1. 1. 1. 1. 0.]
 [0. 0. 1. 1. 0.]
 [1. 0. 0. 0. 1.]]


In [14]:
new_X = A_tilde @ X 
print(new_X)

[[ 1.17555069  4.71751197  3.57427979  3.93145679]
 [-0.72057132  2.13226516  3.78892442  4.0900674 ]
 [-1.05797008  2.95144078  5.5119291   4.44214509]
 [-0.51189169  1.87841944  2.21158888  2.86112046]
 [ 1.62616794  3.53082583  1.32621635  0.47336751]]


# レイヤークラス定義

In [ ]:
class GCNlayer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        Adjacency_matrix: torch.Tensor, # 隣接行列
        ):

        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

        # add I matrix to Adjacency_matrix
        self.A_tilde = Adjacency_matrix + torch.eye(Adjacency_matrix.size(-2)) # GCNは自己ループを必要とする。
